In [31]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

In [32]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.

In [33]:
simulator = BasicSimulator()

# Function to generate random bit using quantum circuit
def get_quantum_random_bit():
    qc = QuantumCircuit(1, 1)
    qc.h(0)
    qc.measure(0, 0)
    compiled_circuit = transpile(qc, simulator)
    job = simulator.run(compiled_circuit, shots=1)
    result = job.result()
    counts = result.get_counts(qc)

    # Extract the measured bit ('0' or '1') and convert to an integer
    measured_bit = list(counts.keys())[0]
    return int(measured_bit)

def generate_random_sequence(length):
    return [get_quantum_random_bit() for _ in range(length)]

In [34]:
# ==========================================
# ALICE
# ==========================================
def encode_message(bits, bases):
    encoded_qubits = []

    for bit, basis in zip(bits, bases):
        qc = QuantumCircuit(1, 1)
        if bit == 1:
            qc.x(0) # Flips |0> to |1>
        if basis == 1:
            qc.h(0) # Changes basis (|0>, |1>) to (|+>, |->)

        encoded_qubits.append(qc)

    return encoded_qubits

# --- Alice Steps ---
# Define initial sequence of 20 qubits
sequence_length = 20

# Generate Alice random bits
alice_bits = generate_random_sequence(sequence_length)

# Generate Alice random bases
alice_bases = generate_random_sequence(sequence_length)

# Encodes Alice qubits
message_qubits = encode_message(alice_bits, alice_bases)

print("Alice Raw Bits:", alice_bits)
print("Alice Bases:", alice_bases)

Alice Raw Bits: [0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1]
Alice Bases: [0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0]


In [35]:
# ==========================================
# BOB
# ==========================================
def measure_message(message_qubits, bases):
    results = []
    simulator = BasicSimulator()

    for qc, basis in zip(message_qubits, bases):
        if basis == 1:
            qc.h(0) # applies H gate to rotate back to diagonal

        qc.measure(0, 0)
        compiled_circuit = transpile(qc, simulator)
        job = simulator.run(compiled_circuit, shots=1)
        result = job.result()
        counts = result.get_counts(qc)
        measured_bit = int(list(counts.keys())[0])
        results.append(measured_bit)

    return results

# --- Bob Steps ---
# Generate Bob random bases
bob_bases = generate_random_sequence(sequence_length)
# Measure the qubits received from Alice
bob_bits = measure_message(message_qubits, bob_bases)

print("Bob Bases:", bob_bases)
print("Bob Measured Bits:", bob_bits)

Bob Bases: [1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1]
Bob Measured Bits: [1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0]


In [36]:
# ==========================================
# ALICE & BOB Communication
# ==========================================
def discard_wrong_value(alice_bases, bob_bases, bits):
    sifted_key = []
    for i in range(len(alice_bases)):
        if alice_bases[i] == bob_bases[i]:
            sifted_key.append(bits[i])
    return sifted_key

alice_key = discard_wrong_value(alice_bases, bob_bases, alice_bits)
bob_key = discard_wrong_value(alice_bases, bob_bases, bob_bits)

print("Alice Key:", alice_key)
print("Bob Key:", bob_key)

if alice_key == bob_key:
    print("Keys matched")
    print(f"Final Key Length: {len(alice_key)} bits (started with {sequence_length} bits)")
else:
    print("Error! Keys do not match")

Alice Key: [0, 1, 0, 0, 1, 1, 0, 1, 1, 1]
Bob Key: [0, 1, 0, 0, 1, 1, 0, 1, 1, 1]
Keys matched
Final Key Length: 10 bits (started with 20 bits)
